In [55]:
import numpy as np
import networkx as nx
import pickle
import os
import json

from utils import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

setup_environment()
set_paper_style()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
class Config:
    DATA_DIR = "data"


# df_a = pd.read_csv(f"{Config.DATA_DIR}/SSDSE-A-2025.csv")
# df_b = pd.read_csv(f"{Config.DATA_DIR}/SSDSE-B-2025.csv")
# df_c = pd.read_csv(f"{Config.DATA_DIR}/SSDSE-C-2025.csv")
# df_d = pd.read_csv(f"{Config.DATA_DIR}/SSDSE-D-2023.csv")
# df_e = pd.read_csv(f"{Config.DATA_DIR}/SSDSE-E-2025.csv")
# df_f = pd.read_csv(f"{Config.DATA_DIR}/SSDSE-F-2023v3.csv")

In [58]:
import pandas as pd
import json
import numpy as np


def create_enhanced_knowledge_graph(csv_file_path, output_file_path):
    """
    より詳細なナレッジグラフJSONを作成
    """
    # データ読み込み
    df = pd.read_csv(csv_file_path)

    # ヘッダー行を取得
    header_row = df.iloc[1]
    data_df = df.iloc[2:].reset_index(drop=True)
    data_df.columns = header_row.values

    # 年度情報を取得（2行目）
    year_row = df.iloc[0]

    nodes = []
    edges = []

    # 都道府県ノードを追加
    prefectures = data_df["都道府県"].dropna().unique()
    for prefecture in prefectures:
        nodes.append({"id": prefecture, "type": "都道府県", "label": prefecture})

    # 市区町村ノードを追加
    for idx, row in data_df.iterrows():
        if pd.notna(row["地域コード"]) and pd.notna(row["市区町村"]):
            properties = {
                "地域コード": row["地域コード"],
                "都道府県": row["都道府県"],
                "市区町村": row["市区町村"],
            }

            # 統計データを年度付きで追加
            for col_idx, col_name in enumerate(data_df.columns):
                if col_name not in ["地域コード", "都道府県", "市区町村"]:
                    value = row[col_name]
                    if pd.notna(value) and value != "":
                        # 年度情報を取得
                        year = (
                            year_row.iloc[col_idx]
                            if col_idx < len(year_row)
                            else "不明"
                        )
                        if pd.notna(year) and year != "":
                            property_key = f"{year}_{col_name}"
                        else:
                            property_key = col_name

                        # 数値の場合は数値型に変換
                        try:
                            if (
                                isinstance(value, str)
                                and value.replace(".", "").replace("-", "").isdigit()
                            ):
                                properties[property_key] = (
                                    float(value) if "." in value else int(value)
                                )
                            else:
                                properties[property_key] = value
                        except:
                            properties[property_key] = value

            nodes.append(
                {
                    "id": row["地域コード"],
                    "type": "市区町村",
                    "label": row["市区町村"],
                    "properties": properties,
                }
            )

            # エッジを追加
            if pd.notna(row["都道府県"]):
                edges.append(
                    {
                        "source": row["地域コード"],
                        "target": row["都道府県"],
                        "label": "IS_IN",
                    }
                )

    knowledge_graph = {"nodes": nodes, "edges": edges}

    # JSONファイルに保存
    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(knowledge_graph, f, ensure_ascii=False, indent=2)

    return knowledge_graph


# 実行例
result = create_enhanced_knowledge_graph(
    "data/SSDSE-A-2025.csv", "knowledge_graph_enhanced.json"
)
print(f"作成完了: ノード数={len(result['nodes'])}, エッジ数={len(result['edges'])}")

作成完了: ノード数=1788, エッジ数=1741


In [4]:
import pandas as pd
df_a = pd.read_csv("data/SSDSE-A-2025.csv")
# row col size
print(df_a.shape)

(1743, 128)
